In [11]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from pydantic import SecretStr,BaseModel , Field 
from typing import TypedDict,NotRequired,Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver # ram wali memory

from dotenv import load_dotenv

load_dotenv()

True

In [12]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    api_key = SecretStr(os.environ["GOOGLE_API_KEY"]),
)

In [13]:
class JokeState(TypedDict):
    topic : str
    joke : NotRequired[str]
    explaination  : NotRequired[str]
    

In [14]:
def generate_joke(state:JokeState):

    prompt = f"Generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).text
    
    return {
        'joke':response
    }

In [15]:
def generate_explaination(state : JokeState):

    prompt = f" write a explaination fro the joke - {state['joke']}"
    response = llm.invoke(prompt).text

    return {'explaination' : response}

In [16]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)


# you make that your bot or workflow maintains persistent 
checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)



In [17]:
config1 = {
    "configurable" : { "thread_id" : "1"}
}
result = workflow.invoke({'topic':'programming'},config=config1)

result

{'topic': 'programming',
 'joke': "There are 10 types of people in the world: those who understand binary, and those who don't.",
 'explaination': 'Here is an explanation of the joke:\n\n### **The Joke:**\n> *"There are 10 types of people in the world: those who understand binary, and those who don\'t."*\n\n---\n\n### **The Explanation:**\n\nTo understand the joke, you need to know how **binary code** works. \n\n1. **What is binary?** \n   Binary is the fundamental language of computers. It uses only two digits: **`0`** and **`1`**. \n\n2. **How does counting work in binary?**\n   Just like our standard counting system (base-10) uses columns for ones, tens, hundreds, etc., binary uses columns that double in value (ones, twos, fours, eights, etc.). \n   * In binary, the number **`0`** is zero.\n   * In binary, the number **`1`** is one.\n   * To write the number **`2`**, you move to the next column. So, the number 2 in binary is written as **`10`** (one "two" and zero "ones").\n\n3. **T

In [18]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'programming', 'joke': "There are 10 types of people in the world: those who understand binary, and those who don't.", 'explaination': 'Here is an explanation of the joke:\n\n### **The Joke:**\n> *"There are 10 types of people in the world: those who understand binary, and those who don\'t."*\n\n---\n\n### **The Explanation:**\n\nTo understand the joke, you need to know how **binary code** works. \n\n1. **What is binary?** \n   Binary is the fundamental language of computers. It uses only two digits: **`0`** and **`1`**. \n\n2. **How does counting work in binary?**\n   Just like our standard counting system (base-10) uses columns for ones, tens, hundreds, etc., binary uses columns that double in value (ones, twos, fours, eights, etc.). \n   * In binary, the number **`0`** is zero.\n   * In binary, the number **`1`** is one.\n   * To write the number **`2`**, you move to the next column. So, the number 2 in binary is written as **`10`** (one "two" and zero

In [20]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'programming', 'joke': "There are 10 types of people in the world: those who understand binary, and those who don't.", 'explaination': 'Here is an explanation of the joke:\n\n### **The Joke:**\n> *"There are 10 types of people in the world: those who understand binary, and those who don\'t."*\n\n---\n\n### **The Explanation:**\n\nTo understand the joke, you need to know how **binary code** works. \n\n1. **What is binary?** \n   Binary is the fundamental language of computers. It uses only two digits: **`0`** and **`1`**. \n\n2. **How does counting work in binary?**\n   Just like our standard counting system (base-10) uses columns for ones, tens, hundreds, etc., binary uses columns that double in value (ones, twos, fours, eights, etc.). \n   * In binary, the number **`0`** is zero.\n   * In binary, the number **`1`** is one.\n   * To write the number **`2`**, you move to the next column. So, the number 2 in binary is written as **`10`** (one "two" and zer